* Model Structure
    - I chose to use a pretrained GPT-2 model feeding into a MLP layer for the sentiment classification. The MLP layer uses a ReLU activation function and dropout layer for regularization. 

* How did you handle the small and imbalanced training set?
    -  I tried several different techniques to approach the small and imbalanced training set, between K-Fold Cross Validation, oversampling, weighting the minority set higher in the cross-entropy loss function, as well as freezing the weights on the GPT-2 portion of the model to help prevent overfitting on the small training set. I even tried a combination of oversampling and weighting the cross-entropy to see if I could obtain a better result than simply using one or the other.\
    \
    Ultimately the model using just the weighted cross entropy ended up being the most stable and well performing on the validation set of the ones I tested. It did end up being overfitted, as you can tell by how it favors the positive class over the negative class, but the weighted cross entropy seemed to help the issue the best of the various methods I tried.

* Key training techniques
    - I kept the learning rate for the GPT-2 transformer portion of the model low at 2e-5 since with such a large model and the very small training set the risk of overfitting was very large. The learning rate for the MLP layer that uses the final hidden state of the transformer could be higher at 1e-3. I wanted to have a higher batch size to make the gradient smoother and less jittery, but unfortunately my laptop with 16GB RAM runs out of memory if the batch size is any larger than 6. I used AdamW for the optimizer since it is supposed to be better for regularization than the regular Adam optimizer, and I was trying to avoid overfitting the model.

#### Evaluation Total Accuracy & Confusion Matrix ####
    - 66% accuracy on the public test set

Confusion matrix:

         [[ 83 117]
         [ 20 180]]

Classification report:

              precision    recall  f1-score   support

    negative       0.81      0.41      0.55       200
    positive       0.61      0.90      0.72       200

    accuracy                           0.66       400
    macro avg      0.71      0.66      0.64       400
    weighted avg   0.71      0.66      0.64       400

In [5]:
import torch
import csv
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import GPT2Tokenizer, GPT2Model
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
class GPT2Classifier(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=2):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")  # NOT frozen this time
        gpt2_dim = self.gpt2.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(gpt2_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, enc):
        out = self.gpt2(**enc)
        hidden = out.last_hidden_state  # (batch, seq_len, hidden_dim)

        last_idx = enc["attention_mask"].sum(dim=1) - 1
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[batch_idx, last_idx]  # (batch, hidden_dim)

        return self.head(pooled)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def collate_fn(batch):
    ids, texts, labels = zip(*batch)
    enc = tokenizer(list(texts), return_tensors="pt", padding=True,
                     truncation=True, max_length=512)
    return list(ids), enc, torch.tensor(labels, dtype=torch.long)


class ReviewDataset(Dataset):
    def __init__(self, ids, texts, labels, max_len=512):
        self.ids = ids
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
 
    def __len__(self):
        return len(self.texts)
 
    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx], self.labels[idx]


def load_data(csv_path):
    ids, texts, labels = [], [], []
    with open(csv_path, encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        next(reader)  # skip header row
        for row in reader:
            ids.append(row[0])
            texts.append(row[1])
            labels.append(int(row[2]))
    return ids, texts, labels


def write_predictions_csv(ids, preds, out_path):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "predicted_label"])
        for id_, pred in zip(ids, preds):
            writer.writerow([id_, int(pred)])
    print(f"Wrote {len(ids)} predictions to {out_path}")

def get_predictions(model, loader):
    """Run the model over a loader and return (ids, true_labels, predicted_labels)."""
    model.eval()
    all_ids, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for ids, enc, yb in tqdm(loader, desc="Evaluating model on test set", unit="batch"):
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            preds = model(enc).argmax(dim=1).cpu().numpy()
            all_ids.extend(ids)
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    return all_ids, all_labels, all_preds


BATCH_SIZE = 6
SAVE_PATH = "model_checkpoint/5_epoch_fine_tune_sentiment_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_ids, test_texts, test_labels = load_data("data/hidden_test_with_labels.csv")

test_loader = DataLoader(ReviewDataset(test_ids, test_texts, test_labels),
                              batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


model = GPT2Classifier().to(DEVICE)
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))

eval_ids, eval_labels, eval_preds = get_predictions(model, test_loader)

Evaluating model on test set: 100%|██████████| 67/67 [02:33<00:00,  2.29s/batch]


### Generate Confusion Matrix & Report ###

In [3]:
print("Confusion matrix:\n")
print(confusion_matrix(eval_labels, eval_preds))
print("\nClassification report:\n")
print(classification_report(eval_labels, eval_preds, target_names=["negative", "positive"]))

Confusion matrix:

[[ 83 117]
 [ 20 180]]

Classification report:

              precision    recall  f1-score   support

    negative       0.81      0.41      0.55       200
    positive       0.61      0.90      0.72       200

    accuracy                           0.66       400
   macro avg       0.71      0.66      0.64       400
weighted avg       0.71      0.66      0.64       400



### Generate CSV ###

In [ ]:
write_predictions_csv(eval_ids, eval_preds, "hidden_test_predictions.csv")

### Use of AI ###

I used Claude Sonnet 5 to write most of the code for my model and made adjustments on specific hyperparameters and settings manually myself. Since training the model was very time consuming, it was helpful to be able to quickly iterate and experiment with different strategies to see which would be successful. 

I also consulted Claude for feedback on conclusions I was drawing on model results or advice on decisions I was considering regarding the model. If I was considering following its advice, I tried to corroborate what it was saying with machine learning advice given online, such as when I was trying to figure out the best method for dealing with the class imbalance problem.

Here are the prompts sent to the AI in the process of generating the final model for the submission:

<b>Prompt 1:</b>
I need to build a sentiment classifier for movie reviews. For this assignment I am given a small portion of the Pang and Lee movie review polarity dataset that I must use for training, containing 180 positive and 60 negative reviews. I am given a test set with 200 positive and 200 negative reviews. I am allowed to use any of the models we studied in the course (explicitly mentioned are RNNs, Transformers, pretrained embeddings, and pretrained language models). One of the key challenges described in the assignment is to navigate overfitting, class imbalance, and evaluation tokens not shown in the training data. Here's a more explicit set of rules:

    You may use standard Python machine learning libraries, including PyTorch, scikit-learn, Hugging Face Transformers, NLTK, NumPy, and pandas.
    You may use pretrained models or pretrained embeddings.

My judgement based on the lectures and the assignment is that my best approach is to take a pretrained decoder and attach MLP layers and softmax to fine tune it to perform sentiment analysis on the movie reviews. I think using a pretrained model is important given our very small dataset.

What are your thoughts on this approach? If I were to continue in this direction, how should I manage having 3 times as many negative reviews as positive reviews? I would appreciate your analysis.

<b>Prompt 2:</b>
For what it's worth, I think a fine tuned decoder (specifically GPT-2) is the method the professor would probably recommend among the methods listed. In one of the lectures they specifically mentioned a decoder with a MLP layer and softmax used for emotion classification and said they were considering that for an assignment before giving the instructions I've already provided. I'm also going to be graded in the second part on a secret test set that will not be provided until I've finalized my model. Ideally I would also like something that will not take a ton of time to implement.

<b>Prompt 3:</b>
Can you generate something a little more barebones that would be a starting point for me to work off of? It's hard to digest so much code at once so I'd rather have something a bit more straightforward that I can develop as I go.

<b>Prompt 4:</b>
So as far as I understand it, what the below lines 

    train_feats = get_features(train_texts).to(DEVICE)
    val_feats = get_features(val_texts).to(DEVICE)
    test_feats = get_features(test_texts).to(DEVICE)

and the get_features method are doing, is prior to training the model at all, we take each possible sample in all of our datasets and run it through the pretrained GPT-2 model, and get the output. Since we're considering the transformer GPT-2 portion of the model as frozen and not to be updated in backpropagation, we get the final hidden layer for each sample and use that as our input for each time we train or test on that sample, so we don't have to constantly recompute those calculations during training when they will always have the same results. Is that right?

<b>Prompt 5:</b>
Is it possible for you to update the code to save the state of the hidden layers locally so they do not need to be recomputed every time the python script is run?

<b>Prompt 6:</b>
For future reference, you can add the next(reader) after creating the reader for the labels row and the test data file is named "public_test.csv"

The validation accuracy is about .70, which is not the best. I also think intuitively, although the risk of overfitting is greater, it's worth it to not freeze the transformer. Movie reviews have a lot of specific phrases that are very key to whether the review is positive or negative. If someone says "i give thirteen days an a-" or "overall , event horizon is a smart film indeed", those particular phrases are critically important to predicting the sentiment of the review. And the attention mechanism in transformers is good at precisely that kind of thing, intelligently extracting the key information in text. Freezing the transformer prevents us from using that to our advantage at all since the LLM has not been trained to treat those tokens with significantly more weight than others, at least based on my intuition on how our model is currently set up. Let me know your analysis and advice on that and if you can create a version that updates the transformer weights through backpropagation, with a new filename so I can return to the old code if needed.

<b>Prompt 7:</b>
Could you implement the following on the previously generated code?

* weighted loss that penalizes misclassifications of the minority class in the training set to a higher degree to compensate for its smaller presence in the training data
* k-fold cross-validation
* saving the model to disk once training is complete

Don't implement it, but let me also know your thoughts on oversampling to help solve the class imbalance problem. Your previous suggestion of augmentation I think does not work unfortunately, the original assignment says "You may not manually label, rewrite, or modify the training and testing data examples." so it seems fully out. If I were to train on the original negative samples 3x as often, that would preclude using the weighted loss, right? it seems like a mutually exclusive thing. I don't know how to weigh one option vs the other. Let me know what you think.

<b>Prompt 8:</b> This is simply too slow on my laptop, one epoch takes like 6-10 minutes or something like that. Could you remove the cross validation? 

I did run on 3 folds before giving up though, so we have some additional data:

1st run (5 fold)
Fold 1 | Epoch 1 | train loss: 1.0130 | val loss: 0.7250 | val acc: 0.4167
Fold 1 | Epoch 2 | train loss: 0.6858 | val loss: 0.7673 | val acc: 0.3958
Fold 1 | Epoch 3 | train loss: 0.5444 | val loss: 0.7428 | val acc: 0.7083
Fold 1 | Epoch 4 | train loss: 0.3577 | val loss: 0.7799 | val acc: 0.6042
Fold 1 | Epoch 5 | train loss: 0.2208 | val loss: 0.6149 | val acc: 0.7500
Fold 1 best val loss: 0.6149 (val acc at that point: 0.7500)

Fold 2 | Epoch 1 | train loss: 1.3054 | val loss: 0.6339 | val acc: 0.6250
<called it quits here>

2nd run (3 fold)

Fold 1 | Epoch 1 | train loss: 1.2153 | val loss: 0.7694 | val acc: 0.3625
Fold 1 | Epoch 2 | train loss: 0.6716 | val loss: 0.8580 | val acc: 0.5375
Fold 1 | Epoch 3 | train loss: 0.5893 | val loss: 0.8227 | val acc: 0.4875
Fold 1 | Epoch 4 | train loss: 0.4720 | val loss: 0.8202 | val acc: 0.6750
Fold 1 best val loss: 0.7694 (val acc at that point: 0.3625)

Fold 2 | Epoch 1 | train loss: 1.1580 | val loss: 0.8271 | val acc: 0.2875
Fold 2 | Epoch 2 | train loss: 0.7033 | val loss: 0.7744 | val acc: 0.5125
Fold 2 | Epoch 3 | train loss: 0.5734 | val loss: 0.7936 | val acc: 0.4250
Fold 2 | Epoch 4 | train loss: 0.4644 | val loss: 0.6630 | val acc: 0.5375
Fold 2 best val loss: 0.6630 (val acc at that point: 0.5375)

It's interesting that the validation loss is still going down a fair bit by epoch 4 consistently, and even by epoch 5 in the first case. The only major change made besides adding the K-fold CV was the weighted loss, so it seems fair to say it's resulted in a decent improvement. It also seems clear in comparing these numbers to the ones prior to the K-fold implementation that the loss in training data due to one of the folds being used for validation really negatively hurts the model. For the final model I train once I've settled on all the hyperparameters, is it a good idea to not use a validation set at all to give myself extra data? I think it could really help performance and I still get a look at the test set to see how it turned out.

Also, if you were forced to make a guess given all the restrictions and complications involved in this task, as well as the freedom given me to use whatever architecture I want to attack this problem, what would you consider a good final accuracy I should be aiming for?

<b>Prompt 9:</b>
Here is some data running the version without the K-fold cross validation:

Run 1:
Epoch 1 | train loss: 1.1797 | val loss: 1.1469 | val acc: 0.7500
Epoch 2 | train loss: 1.4151 | val loss: 0.7520 | val acc: 0.7083
Epoch 3 | train loss: 0.7189 | val loss: 0.6237 | val acc: 0.7292
Epoch 4 | train loss: 0.5981 | val loss: 0.6535 | val acc: 0.6667
Epoch 5 | train loss: 0.4436 | val loss: 0.6203 | val acc: 0.7917
Epoch 6 | train loss: 0.3396 | val loss: 0.6055 | val acc: 0.6667

Run 2:
Epoch 1 | train loss: 0.9930 | val loss: 0.7087 | val acc: 0.3958
Epoch 2 | train loss: 0.6386 | val loss: 0.6125 | val acc: 0.7083
Epoch 3 | train loss: 0.5135 | val loss: 0.6329 | val acc: 0.6458
Epoch 4 | train loss: 0.3761 | val loss: 0.6626 | val acc: 0.7708
Epoch 5 | train loss: 0.2974 | val loss: 0.6690 | val acc: 0.7917
Epoch 6 | train loss: 0.1512 | val loss: 0.5075 | val acc: 0.7917
Epoch 7 | train loss: 0.0466 | val loss: 0.4374 | val acc: 0.8125
Epoch 8 | train loss: 0.0200 | val loss: 0.6383 | val acc: 0.8125
Epoch 9 | train loss: 0.0146 | val loss: 0.5178 | val acc: 0.8542
Epoch 10 | train loss: 0.0092 | val loss: 0.5849 | val acc: 0.8125

It's seeming like epoch 6 or 7 is probably a better cutoff point now, right? Aside from doing another run up to that point to get more data is there anything else you think I should be considering to do my due diligence to make sure the model is not overfitted or is behaving unexpectedly at that point?

<b>Prompt 10:</b>
Could you have the model print the confusion matrix on the validation set after every epoch? For the final model I make I don't want to set aside any data for the validation set as I mentioned earlier. Since we have so little data I don't want to spend 20% of it on validation for the final model when I generally understand the curve of the model I'm training. So don't add any early stopping functionality. Also could you update the code to only have one run that saves once the full number of epochs is finished rather than a diagnostic and final run? The distinction doesn't really provide any utility given how long these take to train.

<b>Prompt 11:</b>
You misunderstand what I mean. I am doing a final testing run through of the model with the current configuration, including the validation set as I have been, and would like to do it with the confusion matrix on that included validation set printed after every iteration. Then, once I am happy with the model's performance I will do a final run where I set the test_size in train_test_split to 0.0 and it doesn't need to output the confusion matrix for anything then, I will just review the performance on the test set once the model is finished training.

Do you think it makes sense to add a regularization term to the loss to discourage overfitting?

<b>Prompt 12:</b>
Here's the output of a run with the confusion matrices:

Epoch 1 | train loss: 1.0449 | train acc: 0.6458 | val loss: 0.8077 | val acc: 0.6250\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [6 6]\
  [12 24]\
Epoch 2 | train loss: 0.7582 | train acc: 0.6042 | val loss: 0.6844 | val acc: 0.6250\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [3 9]\
  [ 9 27]\
Epoch 3 | train loss: 0.6161 | train acc: 0.7031 | val loss: 0.5765 | val acc: 0.7917\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [9 3]\
  [ 7 29]\
Epoch 4 | train loss: 0.4837 | train acc: 0.7865 | val loss: 0.6118 | val acc: 0.7292\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [8 4]\
  [ 9 27]\
Epoch 5 | train loss: 0.3019 | train acc: 0.9010 | val loss: 0.5473 | val acc: 0.7083\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [11  1]\
  [13 23]\
Epoch 6 | train loss: 0.1737 | train acc: 0.9427 | val loss: 0.8971 | val acc: 0.7500\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [ 2 10]\
  [ 2 34]\
Epoch 7 | train loss: 0.1350 | train acc: 0.9479 | val loss: 0.8703 | val acc: 0.7917\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [4 8]\
  [ 2 34]\
Saved final model weights to gpt2_sentiment_model.pt\
Epoch 1 | train loss: 1.0449 | train acc: 0.6458 | val loss: 0.8077 | val acc: 0.6250\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [6 6]\
  [12 24]\
Epoch 2 | train loss: 0.7582 | train acc: 0.6042 | val loss: 0.6844 | val acc: 0.6250\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [3 9]\
  [ 9 27]\
Epoch 3 | train loss: 0.6161 | train acc: 0.7031 | val loss: 0.5765 | val acc: 0.7917\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [9 3]\
  [ 7 29]\
Epoch 4 | train loss: 0.4837 | train acc: 0.7865 | val loss: 0.6118 | val acc: 0.7292\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [8 4]\
  [ 9 27]\
Epoch 5 | train loss: 0.3019 | train acc: 0.9010 | val loss: 0.5473 | val acc: 0.7083\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [11  1]\
  [13 23]\
Epoch 6 | train loss: 0.1737 | train acc: 0.9427 | val loss: 0.8971 | val acc: 0.7500\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [ 2 10]\
  [ 2 34]\
Epoch 7 | train loss: 0.1350 | train acc: 0.9479 | val loss: 0.8703 | val acc: 0.7917\
  Val confusion matrix (rows=true, cols=predicted; order=[neg, pos]):\
  [4 8]\
  [ 2 34]\

I'm considering stopping it after epoch 5 now. I've compiled all the runs I've done with the current settings between epochs:

Epoch 5 | train loss: 0.3019 | val loss: 0.5473 | val acc: 0.7083
Epoch 5 | train loss: 0.2974 | val loss: 0.6690 | val acc: 0.7917
Epoch 5 | train loss: 0.4436 | val loss: 0.6203 | val acc: 0.7917
Epoch 5 | train loss: 0.3661 | val loss: 0.6104 | val acc: 0.8125

Epoch 6 | train loss: 0.1512 | val loss: 0.5075 | val acc: 0.7917
Epoch 6 | train loss: 0.3396 | val loss: 0.6055 | val acc: 0.6667
Epoch 6 | train loss: 0.2318 | val loss: 0.7483 | val acc: 0.7917
Epoch 6 | train loss: 0.1737 ||val loss: 0.8971 | val acc: 0.7500

Epoch 7 | train loss: 0.0466 | val loss: 0.4374 | val acc: 0.8125
Epoch 7 | train loss: 0.1110 | val loss: 0.8704 | val acc: 0.7917
Epoch 7 | train loss: 0.1350 | val loss: 0.8703 | val acc: 0.7917

The confusion matrix I posted above also seems more trustworthy for the one after epoch 5. 6 and 7 seem like the model is optimizing a lot more based on that there are substantially more positive reviews than negative ones.
